# Real-Time Building Energy Prediction with Spark Structured Streaming

This notebook consumes weather events from Kafka, applies the saved Spark ML pipeline model, and produces real-time building energy consumption predictions. It also creates 6-hour building-level and daily site-level aggregations for downstream visualisation.

## Overview

The notebook extends the batch energy prediction workflow into a streaming analytics pipeline. Incoming weather events are parsed from Kafka, joined with static building metadata, transformed into model-ready features, and passed through the trained Spark ML model.

## Pipeline Context

**Weather Producer → Kafka Topic (`weather_stream`) → Spark Structured Streaming → ML Model → Parquet Outputs → Kafka Topics → Consumer Visualisation**

This notebook is the central processing layer. It converts raw event messages into predictions and publishes multiple analytical outputs for reporting.

## Spark Session Setup

The Spark session is configured for local structured streaming with Kafka support, Melbourne timezone handling, and checkpointing. Checkpoint locations allow streaming queries to recover state and progress between runs.

## SparkSession Configuration

SparkSession acts as the entry point for streaming ingestion, DataFrame transformations, SQL-style processing, ML model loading, and streaming sinks.

In [ ]:
# ============================================
# STEP 1: CREATE SPARKSESSION (WITH KAFKA SUPPORT)
# ============================================
from pyspark.sql import SparkSession

# Create SparkSession with all required configurations
spark = (SparkSession
    .builder
    # Application identifier for monitoring
    .appName("BuildingEnergyStreaming")
    
    # Use 4 local cores for reproducible streaming execution
    .master("local[4]")
    
    # Set Melbourne timezone for timestamp operations
    .config("spark.sql.session.timeZone", "Australia/Melbourne")
    
    # Checkpoint location for streaming query state persistence
    .config("spark.sql.streaming.checkpointLocation", "/tmp/spark_checkpoint")
    
    # Auto-cleanup temporary checkpoints on restart
    .config("spark.sql.streaming.forceDeleteTempCheckpointLocation", "true")
    
    # Memory allocation (4GB for driver and executors)
    .config("spark.driver.memory", "4g")
    .config("spark.executor.memory", "4g")
    
    # Reduce shuffle partitions for better local performance
    .config("spark.sql.shuffle.partitions", "4")
    
    # CRITICAL: Add Kafka connector package for streaming integration
    # Format: org.apache.spark:spark-sql-kafka-0-10_SCALA_VERSION:SPARK_VERSION
    .config("spark.jars.packages", 
            "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0")
    
    # Create or get existing session
    .getOrCreate()
)

# Reduce log verbosity (only show warnings and errors)
spark.sparkContext.setLogLevel("WARN")

# Display configuration summary
print("="*70)
print("SparkSession Created with Kafka Support!")
print("="*70)
print(f"Application Name: {spark.sparkContext.appName}")
print(f"Cores: 4")
print(f"Timezone: Australia/Melbourne")
print(f"Checkpoint: /tmp/spark_checkpoint")
print(f"Kafka Support: ENABLED")
print("="*70)

## Configuration Constants

Paths, Kafka topics, checkpoint directories, and streaming parameters are defined in one location for easier maintenance and reproducibility.

In [ ]:
# ============================================
# CONFIGURATION CONSTANTS
# ============================================

# Kafka connection settings
KAFKA_BROKER = 'host.docker.internal:9092'  # Kafka broker address (Docker to host)
KAFKA_TOPIC_INPUT = 'weather_stream'         # Input topic from producer notebook producer
KAFKA_TOPIC_6H = 'predictions_6h'            # Output topic for 6-hour aggregations
KAFKA_TOPIC_DAILY = 'predictions_daily'      # Output topic for daily aggregations

# File paths - UPDATE THESE TO YOUR PATHS!
BUILDING_INFO_PATH = 'new_building_information.csv'  # Static building metadata for streaming prediction
MODEL_PATH = 'models/final_random_forest_model'              # Saved batch-trained model
# Example: MODEL_PATH = 'models/final_random_forest_model'

# Checkpoint locations for different streaming queries
# Each query needs its own checkpoint directory for state management
CHECKPOINT_BASE = '/tmp/spark_streaming_checkpoints'
CHECKPOINT_6H = f'{CHECKPOINT_BASE}/6h_aggregation'
CHECKPOINT_DAILY = f'{CHECKPOINT_BASE}/daily_aggregation'
CHECKPOINT_PARQUET_6H = f'{CHECKPOINT_BASE}/parquet_6h'
CHECKPOINT_PARQUET_DAILY = f'{CHECKPOINT_BASE}/parquet_daily'
CHECKPOINT_KAFKA_6H = f'{CHECKPOINT_BASE}/kafka_6h'
CHECKPOINT_KAFKA_DAILY = f'{CHECKPOINT_BASE}/kafka_daily'

# Parquet output paths for persistent storage (Parquet persistence stage)
PARQUET_6H_PATH = '/tmp/predictions_6h_parquet'
PARQUET_DAILY_PATH = '/tmp/predictions_daily_parquet'

print("Configuration set!")
print(f"Kafka Broker: {KAFKA_BROKER}")
print(f"Input Topic: {KAFKA_TOPIC_INPUT}")
print(f"Model Path: {MODEL_PATH}")

## Import Libraries

The notebook imports PySpark functions for schema definition, Kafka ingestion, JSON parsing, timestamp handling, feature engineering, aggregation, and Spark ML pipeline model loading.

In [ ]:
# ============================================
# IMPORT ALL REQUIRED FUNCTIONS
# ============================================
from pyspark.sql.types import (
    StructType, StructField,      # Schema definition
    IntegerType, StringType,       # Data types
    TimestampType, DoubleType
)
from pyspark.sql.functions import (
    # JSON parsing
    from_json, col,
    
    # Time functions
    from_unixtime, current_timestamp,
    hour, dayofweek, month, dayofmonth, year,
    
    # Window functions
    window,
    
    # Aggregations
    sum as spark_sum, avg, count
)

print("All functions imported successfully!")

In [ ]:
# ============================================
# IMPORT ALL LIBRARIES (COMPLETE VERSION)
# ============================================
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField, 
    IntegerType, StringType, TimestampType, DoubleType
)
from pyspark.sql.functions import (
    # JSON parsing
    col, from_json,
    
    # Time conversion
    from_unixtime, current_timestamp,
    
    # Date/Time extraction
    hour, dayofweek, month, dayofmonth, year,
    to_date, to_timestamp,
    
    # String functions
    concat_ws, lpad,
    
    # Math functions
    log1p, sin, cos, radians, lit,
    
    # Conditional
    when,
    
    # Window functions
    window,
    
    # Aggregations
    avg, sum as spark_sum, count
)
from pyspark.ml import PipelineModel
import os

print("ALL libraries imported successfully!")

## Schema Definition and Static Data Loading

Schemas are defined explicitly for weather events and static building metadata. Explicit schemas improve reliability, avoid costly inference, and ensure the streaming features match the trained model inputs.

## Schema Definition Overview

Structured Streaming requires a known schema to parse JSON messages consistently. The schema captures weather features, site identifiers, and event timestamps used for downstream transformations.

In [ ]:
# ============================================
# STEP 2: DEFINE SCHEMAS
# ============================================
from pyspark.sql.types import (
    StructType, StructField, 
    IntegerType, StringType, TimestampType, DoubleType
)

# Weather schema - matches Kafka producer output from producer notebook
# This schema MUST match the JSON structure sent by the producer
weather_schema = StructType([
    # Site identifier (foreign key to buildings table)
    StructField("site_id", IntegerType(), False),
    
    # Original timestamp from weather.csv
    StructField("timestamp", TimestampType(), False),
    
    # Weather measurements (some fields can be NULL)
    StructField("air_temperature", DoubleType(), True),
    StructField("cloud_coverage", IntegerType(), True),
    StructField("dew_temperature", DoubleType(), True),
    StructField("sea_level_pressure", DoubleType(), True),
    StructField("wind_direction", IntegerType(), True),
    StructField("wind_speed", DoubleType(), True),
    
    # Event timestamp added by producer (Unix timestamp in seconds)
    # This is the field we'll use for watermarking!
    StructField("weather_ts", IntegerType(), False)
])

# NEW CODE (CORRECT - has 10 fields to match CSV):
buildings_schema = StructType([
    StructField("site_id", IntegerType(), False),
    StructField("building_id", IntegerType(), False),
    StructField("primary_use", StringType(), False),
    StructField("square_feet", IntegerType(), False),
    StructField("floor_count", IntegerType(), True),
    StructField("row_id", IntegerType(), True),           # ← NEW LINE ADDED!
    StructField("year_built", IntegerType(), True),
    StructField("latent_y", DoubleType(), True),
    StructField("latent_s", DoubleType(), True),
    StructField("latent_r", DoubleType(), True)
])

# Display schema information
print("Schemas defined!")
print(f"\nWeather schema has {len(weather_schema.fields)} fields:")
for field in weather_schema.fields:
    print(f"  - {field.name}: {field.dataType}")
    
print(f"\nBuildings schema has {len(buildings_schema.fields)} fields:")
for field in buildings_schema.fields:
    print(f"  - {field.name}: {field.dataType}")
    
print("\nSTEP 2 COMPLETED!")

## Load Static Building Data

Building metadata is loaded as a static DataFrame and later joined with streaming weather events. This enriches each weather event with building-level attributes required by the prediction model.

In [ ]:
# ============================================
# LOAD BUILDING INFORMATION (STATIC DATA)
# ============================================

# NEW CODE (add .drop("row_id") at the end):
buildings_df = (spark
    .read
    .format("csv")
    .option("header", True)
    .schema(buildings_schema)
    .load(BUILDING_INFO_PATH)
    .drop("row_id")  # ← NEW LINE ADDED! Remove row_id column
)
# Display summary
print("Building information loaded!")
print(f"Total buildings: {buildings_df.count()}")
print("\nFirst 3 buildings:")
buildings_df.show(3, truncate=False)

print("\nSTEP 2 COMPLETED!")

## Kafka Stream Ingestion

This section subscribes to the `weather_stream` Kafka topic and reads weather events as a streaming DataFrame. Each Kafka message is initially received as bytes and then converted into structured JSON columns.

## Kafka Streaming Overview

Spark Structured Streaming continuously reads messages from the Kafka topic and represents each micro-batch as a DataFrame. This enables real-time transformations using familiar Spark DataFrame operations.

## Streaming Data Flow

`Kafka Topic (weather_stream)` → `Raw Kafka DataFrame` → `Parsed Weather Events` → `Feature Engineering` → `Model Predictions`

## Structured Streaming DataFrame

After ingestion, each incoming Kafka record is parsed into typed columns such as `site_id`, weather measurements, and `weather_ts`. This structured representation is required for model feature preparation.

## Parsing JSON from Kafka

Kafka values are decoded from binary to string and parsed using the predefined weather schema. The parsed data becomes a typed streaming DataFrame ready for transformation.

In [ ]:
# ============================================
# STEP 3: INGEST STREAMING DATA FROM KAFKA
# ============================================

# Read from Kafka topic
kafka_stream = (spark
    .readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_BROKER)
    .option("subscribe", KAFKA_TOPIC_INPUT)
    .option("startingOffsets", "latest")  # Start from latest messages
    .option("failOnDataLoss", "false")    # Handle data loss gracefully
    .load()
)

print("Kafka stream configured!")
print("\nKafka stream schema:")
kafka_stream.printSchema()

# Parse JSON from Kafka value field
weather_stream = (kafka_stream
    .select(
        from_json(
            col("value").cast("string"), 
            weather_schema
        ).alias("data")
    )
    .select("data.*")
)

print("\nWeather stream parsed!")
print("\nWeather stream schema:")
weather_stream.printSchema()

print("\nSTEP 3 COMPLETED!")

## Event-Time Watermarking

A 5-second watermark is applied to weather event time. This controls how late-arriving records are handled and helps Spark manage state for time-based streaming operations.

## Watermarking Concept

The watermark tracks the latest observed event time and allows Spark to discard records that arrive beyond the configured lateness threshold.

## Watermark Formula

`Current Watermark = Maximum Event Time - 5 Seconds`

In [ ]:
# ============================================
# STEP 4: APPLY WATERMARK
# ============================================
# Requirement: Discard data received more than 5 seconds late

# Step 4.1: Convert weather_ts (Unix timestamp in seconds) to timestamp type
# weather_ts is an Integer (e.g., 1730630400)
# from_unixtime converts it to a proper Timestamp column
weather_with_timestamp = weather_stream.withColumn(
    "event_time",  # Create new column for event timestamp
    from_unixtime(col("weather_ts")).cast("timestamp")
    # from_unixtime: Unix timestamp → timestamp
    # Example: 1730630400 → "2024-11-03 12:00:00"
)

# Step 4.2: Apply watermark on event_time
# Watermark = MAX(event_time) - 5 seconds
# Any event with event_time < watermark will be DROPPED
weather_watermarked = weather_with_timestamp.withWatermark(
    "event_time",   # Column to apply watermark on
    "5 seconds"     # Late tolerance threshold (project requirement)
)

print("Watermark applied!")
print("  Late data tolerance: 5 seconds")
print("  Data received >5 seconds late will be discarded")
print("\nWatermarked stream schema:")
weather_watermarked.printSchema()

print("\nSTEP 4 COMPLETED!")

## Streaming Feature Transformation

The streaming weather data is transformed to match the feature structure used by the saved batch-trained model. This includes time-window alignment, seasonal flags, weather aggregation, and building metadata joins.

## Transformation Overview

The transformation layer converts raw streaming weather records into model-ready rows. It aligns events to 6-hour windows and prepares the same predictor columns used during batch training.

## Add Season Flag

A peak/off-peak feature is created to represent periods of higher heating or cooling demand. This feature helps the model capture seasonal variation in energy consumption.

In [ ]:
# ============================================
# STEP 5.1: ADD SEASON FLAG
# ============================================
# Replicate batch training workflow feature: season_flag based on month

# Extract month from timestamp and map to season
# Australia's seasons:
#   Summer: Dec-Feb (12, 1, 2)
#   Autumn: Mar-May (3, 4, 5)
#   Winter: Jun-Aug (6, 7, 8)
#   Spring: Sep-Nov (9, 10, 11)
weather_with_season = weather_watermarked.withColumn(
    "month", month(col("timestamp"))  # Extract month (1-12)
).withColumn(
    "season_flag",
    when((col("month") >= 3) & (col("month") <= 5), "autumn")   # Mar-May
    .when((col("month") >= 6) & (col("month") <= 8), "winter")  # Jun-Aug
    .when((col("month") >= 9) & (col("month") <= 11), "spring") # Sep-Nov
    .otherwise("summer")                                         # Dec-Feb
)

print("Season flag added")

## Weather Aggregation Strategy

Weather measurements are aggregated into 6-hour windows to align with the model target variable. This keeps streaming features consistent with the batch feature engineering pipeline.

In [ ]:
# ============================================
# STEP 5.2: AGGREGATE WEATHER TO 6H WINDOWS
# ============================================
# This replicates your batch training workflow aggregation strategy

weather_6h = (weather_with_season
    # Step 1: Extract date and hour from timestamp
    .withColumn("date", to_date(col("timestamp")))       # Extract date (YYYY-MM-DD)
    .withColumn("hour", hour(col("timestamp")))          # Extract hour (0-23)
    
    # Step 2: Calculate 6-hour bucket (0, 1, 2, 3)
    .withColumn("bucket", (col("hour") / 6).cast("int"))
    .withColumn("bucket_start_hour", col("bucket") * 6)
    # Examples:
    #   hour=3  → bucket=0 → start=0  (00:00)
    #   hour=14 → bucket=2 → start=12 (12:00)
    
    # Step 3: Create window_start timestamp
    # Combine date + bucket_start_hour to get window start time
    .withColumn(
        "window_start",
        to_timestamp(
            concat_ws(
                " ",                                     # Separator
                col("date").cast("string"),              # "2024-11-03"
                lpad(col("bucket_start_hour").cast("string"), 2, "0")  # "12"
            )
            # Result: "2024-11-03 12" → converted to timestamp
        )
    )
    
    # Step 4: Aggregate weather metrics per (site_id, window_start)
    # Group by site_id, window_start, and metadata columns
    .groupBy("site_id", "window_start", "season_flag", "event_time", "weather_ts")
    .agg(
        # Calculate average of each weather metric over the 6-hour window
        avg("air_temperature").alias("air_temperature_6h"),
        avg("cloud_coverage").alias("cloud_coverage_6h"),
        avg("dew_temperature").alias("dew_temperature_6h"),
        avg("sea_level_pressure").alias("sea_level_pressure_6h"),
        avg("wind_direction").alias("wind_direction_6h"),
        avg("wind_speed").alias("wind_speed_6h")
    )
)

print("Weather aggregated to 6-hour windows")

## Join Streaming Weather with Static Building Data

The 6-hour weather stream is joined with building metadata using `site_id`. This creates building-specific prediction rows for each weather window.

In [ ]:
# ============================================
# STEP 5.3: JOIN WITH BUILDING INFO
# ============================================

# Perform inner join between streaming weather and static buildings
# Join key: site_id (each site can have multiple buildings)
weather_with_buildings = weather_6h.join(
    buildings_df,       # Static building DataFrame
    on="site_id",       # Join column
    how="inner"         # Inner join: keep only matching site_ids
)

print("Joined with building information")
print(f"Columns after join: {len(weather_with_buildings.columns)}")
# Expected: ~20+ columns (weather + building attributes)

## Feature Engineering for ML Model

Additional predictors are derived for the trained model, including building age, transformed building size, time-block features, and encoded weather variables.

In [ ]:
# ============================================
# FEATURE ENGINEERING FOR STREAMING INFERENCE
# ============================================

features_engineered = (weather_with_buildings
    # Feature 1: Building age (current year - construction year)
    # Older buildings may have different energy patterns
    .withColumn("building_age", lit(2025) - col("year_built"))
    
    # Feature 2: Log-transform square feet
    # log1p(x) = log(1+x) handles zero values safely
    # Reduces impact of very large buildings
    .withColumn("log_square_feet", log1p(col("square_feet")))
    
    # Feature 3 & 4: Circular encoding for wind direction
    # Convert degrees (0-360) to (sin, cos) components
    .withColumn("wind_dir_sin", sin(radians(col("wind_direction_6h"))))
    .withColumn("wind_dir_cos", cos(radians(col("wind_direction_6h"))))
    # radians: degrees → radians (required by sin/cos)
    # sin/cos: create circular encoding
)

print("Features engineered:")
print("  - building_age")
print("  - log_square_feet")
print("  - wind_dir_sin, wind_dir_cos")

In [ ]:
# ============================================
# STEP 5.5: HANDLE MISSING VALUES
# ============================================
# Fill missing values with defaults (must match batch training workflow preprocessing!)

features_filled = features_engineered.fillna({
    # Weather features - use neutral/typical values
    "cloud_coverage_6h": 0,           # Clear sky
    "sea_level_pressure_6h": 1013.25, # Standard pressure (millibars)
    "wind_direction_6h": 0,           # North
    
    # Building features - use safe defaults
    "floor_count": 1,                 # Single-story building
    "year_built": 2000,               # Approximate median
    
    # Latent features - use neutral values
    "latent_y": 0.0,
    "latent_s": 0.0,
    "latent_r": 0.0
})

print("Missing values filled with defaults")

## Select Model Features

Only the columns required by the trained Spark ML pipeline are selected. Keeping the feature set consistent prevents schema mismatch when applying the saved model.

In [ ]:
# ============================================
# STEP 5.6: SELECT FEATURES FOR MODEL
# ============================================
# These are the required features expected by the saved model.
# DO NOT modify this list unless your batch training workflow model is different

final_features = features_filled.select(
    # Identifiers (for tracking and aggregation)
    "site_id",
    "building_id",
    "window_start",
    "event_time",
    "weather_ts",
    
    # Categorical features (encoded by your pipeline's StringIndexer)
    "primary_use",      # Building type
    "season_flag",      # Season
    
    # Numeric features (scaled by your pipeline)
    "log_square_feet",          # Building size (log-transformed)
    "floor_count",              # Building height
    "building_age",             # Age in years
    "air_temperature_6h",       # Temperature (°C)
    "dew_temperature_6h",       # Dew point (°C)
    "sea_level_pressure_6h",    # Pressure (millibars)
    "wind_speed_6h",            # Wind speed (m/s)
    "cloud_coverage_6h",        # Cloud cover (0-8 oktas)
    "wind_dir_sin",             # Wind direction (sin component)
    "wind_dir_cos",             # Wind direction (cos component)
    "latent_y",                 # Latent feature Y
    "latent_s",                 # Latent feature S
    "latent_r"                  # Latent feature R
)

print("="*70)
print(" STEP 5 COMPLETED!")
print("="*70)
print(f"Total features: {len(final_features.columns)}")
print("\nFeature columns:")
for col_name in final_features.columns:
    print(f"  - {col_name}")
print("\nData is ready for your final_random_forest_model model!")

## Preview Transformed Data

This optional preview helps verify that streaming features are generated correctly before the model is applied.

In [ ]:
# ============================================
# OPTIONAL: PREVIEW TRANSFORMED DATA
# ============================================
# Run this to verify transformations are working correctly

print("Starting preview query (will run for 30 seconds)...")

# Create a streaming query that prints to console
query_preview = (final_features
    .writeStream
    .outputMode("append")          # Print new rows as they arrive
    .format("console")             # Output to console
    .option("truncate", False)     # Don't truncate long values
    .option("numRows", 3)          # Show 3 rows per batch
    .trigger(processingTime='5 seconds')  # Process every 5 seconds
    .start()
)

# Let it run for 30 seconds to see a few batches
import time
time.sleep(30)

# Stop the preview query
query_preview.stop()
print(" Preview stopped")
print("\nIf you saw data printed above, transformations are working!")
print("If no data appeared, check:")
print("  1. Producer (producer notebook) is running")
print("  2. Kafka broker is accessible")
print("  3. Topic name is correct")

## Apply the Saved Spark ML Model

The trained pipeline model is loaded from the local `models/` directory and applied to the streaming feature DataFrame to produce energy-consumption predictions.

## Prediction Stage Overview

The streaming model generates predicted 6-hour energy consumption for each building and weather window. These predictions are then used for raw outputs and aggregated reporting.

## Set Model Path

The model path points to the saved Random Forest pipeline from the batch training notebook. Model artifacts are kept outside Git if they are too large.

In [ ]:
# ============================================
# STEP 6.0: SET MODEL PATH
# ============================================
# UPDATE THIS PATH TO YOUR batch training workflow SAVED MODEL!

MODEL_PATH = 'models/final_random_forest_model'
# This should be the path where you saved your best model in batch training workflow

print(f"Model Path: {MODEL_PATH}")

## Load Model

The Spark ML pipeline model is loaded once and reused for streaming inference.

In [ ]:
# ============================================
# STEP 6.1: LOAD YOUR batch training workflow MODEL
# ============================================
from pyspark.ml import PipelineModel

# Load the trained pipeline model from disk
# This model contains all preprocessing + ML algorithm
model = PipelineModel.load(MODEL_PATH)

print("="*70)
print("Model loaded successfully!")
print("="*70)
print(f"Model stages: {len(model.stages)}")
print("\nPipeline stages:")
for i, stage in enumerate(model.stages):
    print(f"  {i+1}. {type(stage).__name__}")
    # Expected stages:
    #   1. StringIndexer (for primary_use)
    #   2. StringIndexer (for season_flag)
    #   3. VectorAssembler
    #   4. Imputer
    #   5. StandardScaler
    #   6. RandomForestRegressionModel (or your chosen algorithm)
    
print("\n Model ready for predictions!")

## Apply Model to Streaming Data

The model transforms each streaming micro-batch into prediction rows containing building identifiers, timestamps, features, and predicted energy values.

In [ ]:
# ============================================
# STEP 6.2: APPLY MODEL TO STREAMING DATA
# ============================================

# Apply the entire pipeline to streaming data
# This performs: encoding → assembling → imputing → scaling → predicting
predictions = model.transform(final_features)

print("Model applied to streaming data!")
print("\nPrediction DataFrame schema (key columns):")
predictions.select("building_id", "site_id", "window_start", "prediction").printSchema()
# Expected output:
# root
#  |-- building_id: integer (nullable = true)
#  |-- site_id: integer (nullable = true)
#  |-- window_start: timestamp (nullable = true)
#  |-- prediction: double (nullable = true)  ← Energy prediction!

## Streaming Prediction Output

This query prints incoming prediction rows to the console for debugging and validation during local development.

In [ ]:
# ============================================
# STEP 6a: PRINT PREDICTIONS AS THEY ARRIVE
# ============================================

query_6a = (predictions
    .select(
        "building_id",
        "site_id", 
        "window_start",
        "event_time",
        "prediction"
    )
    .writeStream
    .outputMode("append")
    .format("console")
    .option("truncate", False)
    .option("numRows", 10)
    .trigger(processingTime='5 seconds')
    .option("checkpointLocation", "/tmp/spark_checkpoint/query_6a")  # ← UNIQUE!
    .start()
)

print("="*70)
print("✓ Query 6a Started: Printing predictions")
print("="*70)
print(f"Query ID: {query_6a.id}")
print(f"Checkpoint: /tmp/spark_checkpoint/query_6a")
print("="*70)

## 6-Hour Building-Level Aggregation

Predictions are aggregated by building and 6-hour window to estimate total energy demand per building over each time block.

## Clear Checkpoint When Required

Checkpoint folders may need to be cleared during local testing if a query schema changes between runs.

In [ ]:
# Run this first if you encounter checkpoint errors
import shutil
shutil.rmtree("/tmp/spark_checkpoint/query_6b", ignore_errors=True)
print("Checkpoint cleared")

## Watermark Configuration Check

This section verifies that event-time handling and aggregation logic are aligned before starting the aggregation stream.

In [ ]:
# ============================================
# FIX: ADD WATERMARK CONFIG TO EXISTING SESSION
# ============================================

# Disable stateful operator correctness check
# This allows aggregations without explicit watermark in groupBy
spark.conf.set("spark.sql.streaming.statefulOperator.checkCorrectness.enabled", "false")

print("Watermark correctness check disabled")
print("Now try running query 6b again!")

## 6-Hour Aggregation Implementation

The precomputed `window_start` column is used as the 6-hour time anchor. Grouping by `building_id`, `site_id`, and `window_start` provides a clean building-level energy summary for each interval.

## Building-Level 6-Hour Aggregation Query

In [ ]:
# ============================================
# STEP 6b: 6-HOUR AGGREGATION BY BUILDING
# ============================================

from pyspark.sql.functions import sum as spark_sum, count

# Disable watermark check (if needed)
spark.conf.set("spark.sql.streaming.statefulOperator.checkCorrectness.enabled", "false")

# Create aggregation
predictions_6h = (predictions
    .groupBy(
        "building_id",
        "site_id",
        "window_start"
    )
    .agg(
        spark_sum("prediction").alias("total_energy_6h"),
        count("*").alias("record_count")
    )
)

# Start streaming query
query_6b = (predictions_6h
    .writeStream
    .outputMode("complete")
    .format("console")
    .option("truncate", False)
    .option("numRows", 20)
    .trigger(processingTime='7 seconds')
    .option("checkpointLocation", "/tmp/spark_checkpoint/query_6b")  # ← UNIQUE!
    .start()
)

print("="*70)
print("✓ Query 6b Started: 6-hour aggregation by building")
print("="*70)
print(f"Query ID: {query_6b.id}")
print(f"Checkpoint: /tmp/spark_checkpoint/query_6b")
print("="*70)

## Daily Site-Level Aggregation

Predictions are aggregated by site and calendar day to support site-level monitoring and downstream visualisation.

In [ ]:
# Run this first if you encounter checkpoint errors
import shutil
shutil.rmtree("/tmp/spark_checkpoint/query_6c", ignore_errors=True)
print("Checkpoint cleared")

In [ ]:
# ============================================
# FIX: ADD WATERMARK CONFIG TO EXISTING SESSION
# ============================================

# Disable stateful operator correctness check
# This allows aggregations without explicit watermark in groupBy
spark.conf.set("spark.sql.streaming.statefulOperator.checkCorrectness.enabled", "false")

print("Watermark correctness check disabled")
print("Now try running query 6c again!")

## Site-Level Daily Aggregation Query

In [ ]:
# ============================================
# STEP 6c: DAILY AGGREGATION BY SITE
# ============================================
from pyspark.sql.functions import to_date, col, sum as spark_sum, count

# Create daily aggregation from the predictions DataFrame
predictions_daily = (predictions
    .withColumn("day", to_date(col("window_start")))  # Extract date from window_start
    .groupBy(
        "site_id",
        "day"
    )
    .agg(
        spark_sum("prediction").alias("total_energy_daily"),
        count("*").alias("record_count")
    )
)

# Start streaming query
query_6c = (predictions_daily
    .writeStream
    .outputMode("complete")  # Complete mode shows all aggregations
    .format("console")
    .option("truncate", False)
    .option("numRows", 20)
    .trigger(processingTime='14 seconds')
    .option("checkpointLocation", "/tmp/spark_checkpoint/query_6c")
    .start()
)

print("="*70)
print("✓ Query 6c Started: Daily aggregation by site")
print("="*70)
print(f"Query ID: {query_6c.id}")
print(f"Checkpoint: /tmp/spark_checkpoint/query_6c")
print("="*70)

## Persist Streaming Outputs to Parquet

Prediction outputs and aggregation results are saved as Parquet streams. Parquet provides efficient columnar storage and allows downstream readers to consume newly written files incrementally.

## Parquet Persistence Overview

Three output layers are stored separately: individual predictions, 6-hour building aggregations, and daily site aggregations. This separation supports different analysis and visualisation use cases.

## Configure Parquet Paths

In [ ]:
# ============================================
# STEP 7: CONFIGURE PARQUET OUTPUT PATHS
# ============================================

# Define separate paths for each type of output
PARQUET_PREDICTIONS_PATH = '/tmp/predictions_parquet'        # 7a: Individual predictions
PARQUET_6H_AGG_PATH = '/tmp/predictions_6h_agg_parquet'      # 7b: 6-hour aggregations
PARQUET_DAILY_AGG_PATH = '/tmp/predictions_daily_agg_parquet' # 7c: Daily aggregations

# Checkpoint locations
CHECKPOINT_7A = '/tmp/spark_checkpoint/parquet_7a'
CHECKPOINT_7B = '/tmp/spark_checkpoint/parquet_7b'
CHECKPOINT_7C = '/tmp/spark_checkpoint/parquet_7c'

print("Parquet output paths configured:")
print(f"  7a (Predictions):    {PARQUET_PREDICTIONS_PATH}")
print(f"  7b (6h Aggregation): {PARQUET_6H_AGG_PATH}")
print(f"  7c (Daily Aggregation): {PARQUET_DAILY_AGG_PATH}")

## Save Individual Predictions to Parquet

Raw prediction rows are written in append mode so each micro-batch adds new records without overwriting existing results.

In [ ]:
# ============================================
# STEP 7a: SAVE INDIVIDUAL PREDICTIONS TO PARQUET
# ============================================
# Saves data from Step 6a (individual predictions)

query_7a = (predictions
    # Select relevant columns for downstream use
    .select(
        "building_id",      # Building identifier
        "site_id",          # Site identifier
        "window_start",     # 6-hour window start time
        "event_time",       # Event timestamp (for watermark)
        "prediction",       # Energy consumption prediction
        "weather_ts"        # Original producer timestamp
    )
    .writeStream
    .outputMode("append")  # Append new predictions as they arrive
    .format("parquet")     # Use Parquet format for efficient storage
    .option("path", PARQUET_PREDICTIONS_PATH)  # Output directory
    .option("checkpointLocation", CHECKPOINT_7A)  # Checkpoint for fault tolerance
    .trigger(processingTime='5 seconds')  # Write every 5 seconds
    .start()
)

print("="*70)
print("✓ raw prediction Parquet stream Started: Saving individual predictions to Parquet")
print("="*70)
print(f"Query ID: {query_7a.id}")
print(f"Output Path: {PARQUET_PREDICTIONS_PATH}")
print(f"Output Mode: append")
print(f"Trigger: Every 5 seconds")
print("="*70)

## Save 6-Hour Aggregations to Parquet

The 6-hour building-level aggregation stream is written to Parquet for downstream reporting and Kafka publishing.

In [ ]:
# ============================================
# STEP 7b: SAVE 6-HOUR AGGREGATIONS TO PARQUET
# ============================================

PARQUET_6H_AGG_PATH = "/tmp/parquet_output/6h_aggregations"
CHECKPOINT_7B = "/tmp/spark_checkpoint/query_7b"

def write_6h_batch(batch_df, batch_id):
    """Write each batch of 6-hour aggregated data to Parquet."""
    if not batch_df.isEmpty():
        try:
            batch_df.write \
                .mode("append") \
                .parquet(PARQUET_6H_AGG_PATH)
            
            row_count = batch_df.count()
            print(f"  ✓ Batch {batch_id}: Wrote {row_count} records to Parquet")
        except Exception as e:
            print(f"  ✗ Batch {batch_id}: Error - {e}")

# Clear old checkpoint
import shutil
shutil.rmtree(CHECKPOINT_7B, ignore_errors=True)
shutil.rmtree(PARQUET_6H_AGG_PATH, ignore_errors=True)
print("✓ Cleared old 7b checkpoint and parquet files")

# Create streaming query - write ALL columns from predictions_6h
query_7b = (predictions_6h
    .writeStream
    .outputMode("complete")
    .foreachBatch(write_6h_batch)
    .option("checkpointLocation", CHECKPOINT_7B)
    .trigger(processingTime='7 seconds')
    .start()
)

print("="*70)
print("✓ 6-hour aggregation Parquet stream Started: Saving 6-Hour Aggregations to Parquet")
print("="*70)
print(f"Query ID: {query_7b.id}")
print(f"Output: {PARQUET_6H_AGG_PATH}")
print(f"Checkpoint: {CHECKPOINT_7B}")
print("="*70)

## Save Daily Aggregations to Parquet

Daily site-level energy summaries are written to Parquet for dashboarding and shortfall/excess analysis.

In [ ]:
# ============================================
# STEP 7c: SAVE DAILY AGGREGATIONS TO PARQUET
# ============================================

PARQUET_DAILY_AGG_PATH = "/tmp/parquet_output/daily_aggregations"
CHECKPOINT_7C = "/tmp/spark_checkpoint/query_7c"

def write_daily_batch(batch_df, batch_id):
    """Write each batch of daily aggregated data to Parquet."""
    if not batch_df.isEmpty():
        try:
            batch_df.write \
                .mode("append") \
                .parquet(PARQUET_DAILY_AGG_PATH)
            
            row_count = batch_df.count()
            print(f"  ✓ Batch {batch_id}: Wrote {row_count} records to Parquet")
        except Exception as e:
            print(f"  ✗ Batch {batch_id}: Error - {e}")

# Clear old checkpoint
import shutil
shutil.rmtree(CHECKPOINT_7C, ignore_errors=True)
shutil.rmtree(PARQUET_DAILY_AGG_PATH, ignore_errors=True)
print("✓ Cleared old 7c checkpoint and parquet files")

# Create streaming query - write ALL columns from predictions_daily
query_7c = (predictions_daily
    .writeStream
    .outputMode("complete")
    .foreachBatch(write_daily_batch)
    .option("checkpointLocation", CHECKPOINT_7C)
    .trigger(processingTime='14 seconds')
    .start()
)

print("="*70)
print("✓ daily aggregation Parquet stream Started: Saving Daily Aggregations to Parquet")
print("="*70)
print(f"Query ID: {query_7c.id}")
print(f"Output: {PARQUET_DAILY_AGG_PATH}")
print(f"Checkpoint: {CHECKPOINT_7C}")
print("="*70)

## Verify Parquet Outputs

This section checks whether Parquet files are being created successfully by the active streaming queries.

In [ ]:
# ============================================
# VERIFY PARQUET FILES ARE BEING CREATED
# ============================================

import time
import os

# Wait for data to be written (adjust time based on your producer speed)
print("Waiting 20 seconds for Parquet files to be created...")
time.sleep(20)

print("\n" + "="*70)
print("PARQUET FILE VERIFICATION")
print("="*70)

# Function to check directory and preview data
def verify_parquet(path, name):
    print(f"\n{name}:")
    print("-" * 70)
    
    if os.path.exists(path):
        print(f"Directory exists: {path}")
        
        # List files
        files = os.listdir(path)
        parquet_files = [f for f in files if f.endswith('.parquet')]
        print(f"  Total items: {len(files)}")
        print(f"  Parquet files: {len(parquet_files)}")
        
        # Read and preview data
        try:
            df = spark.read.parquet(path)
            record_count = df.count()
            print(f"  Total records: {record_count}")
            
            print(f"\n  Preview (first 5 rows):")
            df.show(5, truncate=False)
            
            return True
        except Exception as e:
            print(f"  Error reading Parquet: {e}")
            return False
    else:
        print(f"Directory not found: {path}")
        print("  Data may not have arrived yet, or producer is not running")
        return False

# Verify all three Parquet outputs
results = []
results.append(verify_parquet(PARQUET_PREDICTIONS_PATH, "7a: Individual Predictions"))
results.append(verify_parquet(PARQUET_6H_AGG_PATH, "7b: 6-Hour Aggregations"))
results.append(verify_parquet(PARQUET_DAILY_AGG_PATH, "7c: Daily Aggregations"))

# Summary
print("\n" + "="*70)
print("VERIFICATION SUMMARY")
print("="*70)
print(f"7a (Predictions):    {'PASS' if results[0] else 'FAIL'}")
print(f"7b (6h Aggregation): {'PASS' if results[1] else 'FAIL'}")
print(f"7c (Daily Aggregation): {'PASS' if results[2] else 'FAIL'}")

if all(results):
    print("\n ALL PARQUET OUTPUTS VERIFIED!")
    print(" STEP 7 COMPLETED SUCCESSFULLY!")
else:
    print("\n Some outputs are missing. Troubleshooting:")
    print("  1. Ensure producer notebook producer is running")
    print("  2. Check all Step 6 queries are active")
    print("  3. Wait longer for data to arrive")
    print("  4. Verify Kafka broker is accessible")

## Check Streaming Query Status

The status of active Spark streaming queries is reviewed to confirm that each stream is running or stopped as expected.

In [ ]:
# ============================================
# CHECK STATUS OF ALL STREAMING QUERIES
# ============================================

print("="*70)
print("STREAMING QUERY STATUS CHECK")
print("="*70)

# Get all active queries
active_queries = spark.streams.active
print(f"\nTotal active queries: {len(active_queries)}")

# Check specific queries
queries_to_check = [
    ("Step 6a (Console)", "query_6a"),
    ("Step 6b (Console)", "query_6b"),
    ("Step 6c (Console)", "query_6c"),
    ("raw prediction Parquet stream (Parquet)", "query_7a"),
    ("6-hour aggregation Parquet stream (Parquet)", "query_7b"),
    ("daily aggregation Parquet stream (Parquet)", "query_7c")
]

print("\nIndividual Query Status:")
print("-" * 70)

for name, var_name in queries_to_check:
    try:
        # Get query object from variable name
        query = eval(var_name)
        status = "RUNNING" if query.isActive else "STOPPED"
        print(f"{name:25s} | {status:12s} | ID: {query.id}")
    except NameError:
        print(f"{name:25s} | NOT FOUND | Variable '{var_name}' does not exist")

print("="*70)

# Quick file check
print("\nQuick File Check:")
print("-" * 70)
paths = [
    (PARQUET_PREDICTIONS_PATH, "7a"),
    (PARQUET_6H_AGG_PATH, "7b"),
    (PARQUET_DAILY_AGG_PATH, "7c")
]

for path, label in paths:
    exists = "✓" if os.path.exists(path) else "✗"
    if os.path.exists(path):
        file_count = len([f for f in os.listdir(path) if f.endswith('.parquet')])
        print(f"{label}: {exists} {path} ({file_count} parquet files)")
    else:
        print(f"{label}: {exists} {path} (not found)")

print("="*70)

if len(active_queries) == 6:
    print("\n✅ All 6 queries are running!")
    print("✅ Parquet persistence stage is working correctly!")
else:
    print(f"\  Expected 6 queries, found {len(active_queries)}")
    print("Some queries may have failed. Check error messages above.")

## Publish Parquet Streams to Kafka

The saved Parquet outputs are read back as streaming DataFrames and forwarded to Kafka topics for consumer-side visualisation.

## Streaming Parquet to Kafka

This stage bridges storage and visualisation. Each Parquet stream is converted into JSON records and published to a Kafka topic with a descriptive name.

## Kafka Topic Naming Strategy

Separate topics are used for raw predictions, 6-hour building aggregations, and daily site aggregations so consumers can subscribe only to the data needed for a specific visualisation.

In [ ]:
# ============================================
# STEP 8: CONFIGURE KAFKA OUTPUT TOPICS
# ============================================

print("="*70)
print("STEP 8: STREAMING PARQUET TO KAFKA")
print("="*70)

# Define Kafka broker (same as producer notebook)
KAFKA_BROKER = 'host.docker.internal:9092'

# Define three Kafka output topics (one for each Parquet source)
KAFKA_TOPIC_RAW = 'predictions_raw'         # For individual predictions (from 7a)
KAFKA_TOPIC_6H = 'predictions_6h'           # For 6-hour aggregations (from 7b)
KAFKA_TOPIC_DAILY = 'predictions_daily'     # For daily aggregations (from 7c)

# Checkpoint locations for each streaming query
CHECKPOINT_8A = '/tmp/spark_checkpoint/kafka_8a'
CHECKPOINT_8B = '/tmp/spark_checkpoint/kafka_8b'
CHECKPOINT_8C = '/tmp/spark_checkpoint/kafka_8c'

print(f"\nKafka Broker: {KAFKA_BROKER}")
print(f"\nOutput Topics:")
print(f"  1. Raw Predictions:     {KAFKA_TOPIC_RAW}")
print(f"  2. 6-hour Aggregations: {KAFKA_TOPIC_6H}")
print(f"  3. Daily Aggregations:  {KAFKA_TOPIC_DAILY}")
print("="*70)

In [ ]:
# ============================================
# CLEAR OLD CHECKPOINTS
# ============================================
import shutil

checkpoints_to_clear = [CHECKPOINT_8A, CHECKPOINT_8B, CHECKPOINT_8C]
for cp in checkpoints_to_clear:
    shutil.rmtree(cp, ignore_errors=True)
    
print("\n✓ Cleared Kafka publishing checkpoints")
print("="*70)

## Stream 1: Individual Predictions to Kafka

The individual prediction Parquet stream is converted to JSON and published to the `predictions_raw` Kafka topic.

In [ ]:
# ============================================
# STEP 8.1: STREAM 1 - INDIVIDUAL PREDICTIONS TO KAFKA
# ============================================
# Read from raw prediction Parquet stream Parquet files

print("\nStream 1: Individual Predictions")
print("-" * 70)

# Define schema (must match what was written in raw prediction Parquet stream)
from pyspark.sql.types import StructType, StructField, IntegerType, TimestampType, DoubleType

predictions_schema = StructType([
    StructField("building_id", IntegerType(), True),
    StructField("site_id", IntegerType(), True),
    StructField("window_start", TimestampType(), True),
    StructField("event_time", TimestampType(), True),
    StructField("prediction", DoubleType(), True),
    StructField("weather_ts", IntegerType(), True)
])

# Read Parquet as streaming DataFrame
predictions_stream = (spark
    .readStream
    .format("parquet")              # Read from Parquet files
    .schema(predictions_schema)     # Use explicit schema
    .load(PARQUET_PREDICTIONS_PATH) # Path to 7a output
)

print("Reading from Parquet (7a): Individual predictions")

# Convert to JSON format for Kafka
# Kafka requires: key (STRING) and value (STRING in JSON format)
predictions_json = (predictions_stream
    .selectExpr(
        "CAST(building_id AS STRING) as key",  # Use building_id as message key
        "to_json(struct(*)) AS value"          # Convert all columns to JSON
    )
)

print("Converted to JSON format")

# Write to Kafka topic
query_8a = (predictions_json
    .writeStream
    .outputMode("append")            # Append new rows as they arrive
    .format("kafka")                 # Kafka sink
    .option("kafka.bootstrap.servers", KAFKA_BROKER)  # Kafka broker address
    .option("topic", KAFKA_TOPIC_RAW)                # Destination topic
    .option("checkpointLocation", CHECKPOINT_8A)     # Checkpoint for fault tolerance
    .trigger(processingTime='5 seconds')             # Process every 5 seconds
    .start()
)

print("="*70)
print("Stream 1 Started: Individual Predictions → Kafka")
print("="*70)
print(f"Query ID: {query_8a.id}")
print(f"Source: {PARQUET_PREDICTIONS_PATH}")
print(f"Kafka Topic: {KAFKA_TOPIC_RAW}")
print(f"Trigger: Every 5 seconds")
print(f"Output Mode: append")
print("="*70)

## Stream 2: 6-Hour Aggregations to Kafka

The 6-hour aggregation Parquet stream is forwarded to the `predictions_6h` Kafka topic for building-level monitoring.

In [ ]:
# ============================================
# STEP 8.2: STREAM 2 - 6-HOUR AGGREGATIONS TO KAFKA
# ============================================
# Read from 6-hour aggregation Parquet stream Parquet files (already aggregated!)

print("\nStream 2: 6-Hour Aggregations")
print("-" * 70)

# Define schema (must match what was written in 6-hour aggregation Parquet stream)
from pyspark.sql.types import LongType

predictions_6h_schema = StructType([
    StructField("building_id", IntegerType(), True),
    StructField("site_id", IntegerType(), True),
    StructField("window_start", TimestampType(), True),
    StructField("total_energy_6h", DoubleType(), True),
    StructField("record_count", LongType(), True)
])

# Read ALREADY AGGREGATED Parquet as streaming DataFrame
predictions_6h_stream = (spark
    .readStream
    .format("parquet")                 # Read from Parquet files
    .schema(predictions_6h_schema)     # Use explicit schema
    .load(PARQUET_6H_AGG_PATH)         # Path to 7b output (aggregated data!)
)

print("Reading from Parquet (7b): Already aggregated 6-hour data")
print(" NO re-aggregation needed - using existing aggregations")

# Convert to JSON format for Kafka
predictions_6h_json = (predictions_6h_stream
    .selectExpr(
        "CAST(building_id AS STRING) as key",  # Use building_id as message key
        "to_json(struct(*)) AS value"          # Convert all columns to JSON
    )
)

print("Converted to JSON format")

# Write to Kafka topic
query_8b = (predictions_6h_json
    .writeStream
    .outputMode("append")            # Append mode works for reading aggregated Parquet
    .format("kafka")                 # Kafka sink
    .option("kafka.bootstrap.servers", KAFKA_BROKER)  # Kafka broker address
    .option("topic", KAFKA_TOPIC_6H)                 # Destination topic
    .option("checkpointLocation", CHECKPOINT_8B)     # Checkpoint for fault tolerance
    .trigger(processingTime='7 seconds')             # Process every 7 seconds
    .start()
)

print("="*70)
print("Stream 2 Started: 6-Hour Aggregations → Kafka")
print("="*70)
print(f"Query ID: {query_8b.id}")
print(f"Source: {PARQUET_6H_AGG_PATH}")
print(f"Kafka Topic: {KAFKA_TOPIC_6H}")
print(f"Trigger: Every 7 seconds")
print(f"Output Mode: append")
print("="*70)

## Stream 3: Daily Aggregations to Kafka

The daily site-level aggregation stream is forwarded to the `predictions_daily` Kafka topic for daily reporting and accuracy analysis.

In [ ]:
# ============================================
# STEP 8.3: STREAM 3 - DAILY AGGREGATIONS TO KAFKA
# ============================================
# Read from daily aggregation Parquet stream Parquet files (already aggregated!)

print("\nStream 3: Daily Aggregations")
print("-" * 70)

# Define schema (must match what was written in daily aggregation Parquet stream)
from pyspark.sql.types import DateType

predictions_daily_schema = StructType([
    StructField("site_id", IntegerType(), True),
    StructField("day", DateType(), True),
    StructField("total_energy_daily", DoubleType(), True),
    StructField("record_count", LongType(), True)
])

# Read ALREADY AGGREGATED Parquet as streaming DataFrame
predictions_daily_stream = (spark
    .readStream
    .format("parquet")                    # Read from Parquet files
    .schema(predictions_daily_schema)     # Use explicit schema
    .load(PARQUET_DAILY_AGG_PATH)         # Path to 7c output (aggregated data!)
)

print(" Reading from Parquet (7c): Already aggregated daily data")
print("NO re-aggregation needed - using existing aggregations")

# Convert to JSON format for Kafka
predictions_daily_json = (predictions_daily_stream
    .selectExpr(
        "CAST(site_id AS STRING) as key",  # Use site_id as message key
        "to_json(struct(*)) AS value"      # Convert all columns to JSON
    )
)

print("Converted to JSON format")

# Write to Kafka topic
query_8c = (predictions_daily_json
    .writeStream
    .outputMode("append")            # Append mode works for reading aggregated Parquet
    .format("kafka")                 # Kafka sink
    .option("kafka.bootstrap.servers", KAFKA_BROKER)  # Kafka broker address
    .option("topic", KAFKA_TOPIC_DAILY)              # Destination topic
    .option("checkpointLocation", CHECKPOINT_8C)     # Checkpoint for fault tolerance
    .trigger(processingTime='14 seconds')            # Process every 14 seconds
    .start()
)

print("="*70)
print("Stream 3 Started: Daily Aggregations → Kafka")
print("="*70)
print(f"Query ID: {query_8c.id}")
print(f"Source: {PARQUET_DAILY_AGG_PATH}")
print(f"Kafka Topic: {KAFKA_TOPIC_DAILY}")
print(f"Trigger: Every 14 seconds")
print(f"Output Mode: append")
print("="*70)

## Verify Streaming Queries

All active streaming queries are checked to confirm that prediction, persistence, and Kafka publishing streams are running as expected.

In [ ]:
# ============================================
# STEP 8.4: VERIFY ALL STREAMING QUERIES
# ============================================

print("\n" + "="*70)
print("ALL ACTIVE STREAMING QUERIES")
print("="*70)

# Get all active queries
active_queries = spark.streams.active
print(f"\nTotal active queries: {len(active_queries)}")

# Expected queries
expected_queries = {
    "Step 6a": "query_6a",
    "Step 6b": "query_6b", 
    "Step 6c": "query_6c",
    "raw prediction Parquet stream": "query_7a",
    "6-hour aggregation Parquet stream": "query_7b",
    "daily aggregation Parquet stream": "query_7c",
    "raw prediction stream": "query_8a",
    "6-hour aggregation stream": "query_8b",
    "daily aggregation stream": "query_8c"
}

print("\nQuery Status Check:")
print("-" * 70)

found_count = 0
for name, var_name in expected_queries.items():
    try:
        query = eval(var_name)
        status = "✓ RUNNING" if query.isActive else "✗ STOPPED"
        print(f"{name:12s} ({var_name:10s}) | {status:12s} | ID: {query.id}")
        if query.isActive:
            found_count += 1
    except NameError:
        print(f"{name:12s} ({var_name:10s}) | ✗ NOT FOUND")

print("-" * 70)
print(f"Active queries: {found_count}/9")

if found_count == 9:
    print("\n ALL 9 QUERIES ARE RUNNING!")
else:
    print(f"\n  Expected 9 queries, found {found_count} running")

print("="*70)